In [ ]:
import os
import sys
import time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.io import wavfile
from scipy.signal import resample_poly, stft
from scipy.fftpack import dct
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score
from IPython.display import clear_output

from pynq import Overlay, allocate

TEST_FOLDER = "./test_audio"

TARGET_WORDS = ['yes', 'no', 'on', 'off', 'stop']
LABELS = ['silence', 'unknown', 'yes', 'no', 'on', 'off', 'stop']

N_MFCC = 40
N_FRAMES = 101
N_INPUT = N_MFCC * N_FRAMES
FIXED_SCALE_IN = 1024.0
FIXED_SCALE_OUT = 4096.0

N_FFT = 400
HOP_LENGTH = 160
N_MELS = 40
SAMPLE_RATE = 16000

UNKNOWN_LOGIT_OFFSET = 1.4

overlay = Overlay("final.bit")

dma = overlay.axi_dma_0
my_ip = overlay.myproject_0

input_buffer = allocate(shape=(N_INPUT,), dtype=np.int32)
output_buffer = allocate(shape=(8,), dtype=np.int32)

input_buffer[:] = 0
output_buffer[:] = 0

def create_mel_weight_matrix(num_mel_filter, num_spectrogram_coeffs, sample_rate, low_freq, high_freq):
    def hz_to_mel(hz):
        return 2595.0 * np.log10(1.0 + hz / 700.0)

    def mel_to_hz(mel):
        return 700.0 * (10.0 ** (mel / 2595.0) - 1.0)
    
    low_mel = hz_to_mel(low_freq)
    higher_mel = hz_to_mel(high_freq)
    
    mel_frequencies = np.linspace(low_mel, higher_mel, num_mel_filter + 2)
    frequency_hz = mel_to_hz(mel_frequencies)

    frequency_ratio = (num_spectrogram_coeffs * 2) * frequency_hz / sample_rate
    frequency_idx = np.floor(frequency_ratio).astype(int)
    
    mel_filter_weights = np.zeros((num_spectrogram_coeffs, num_mel_filter))

    for i in range(num_mel_filter):
        start = frequency_idx[i]
        center = frequency_idx[i + 1]
        end = frequency_idx[i + 2]

        for j in range(start, center):
            if center != start:
                mel_filter_weights[j, i] = (j - start) / (center - start)

        for j in range(center, end):
            if end != center:
                mel_filter_weights[j, i] = (end - j) / (end - center)

    return mel_filter_weights

MEL_FILTER_WEIGHTS = create_mel_weight_matrix(
    num_mel_filter=N_MELS,
    num_spectrogram_coeffs=(N_FFT // 2 + 1),
    sample_rate=SAMPLE_RATE,
    low_freq=20.0,
    high_freq=4000.0
)

HANN_WINDOW = np.hanning(N_FFT)
HANN_SUM = np.sum(HANN_WINDOW)

def extract_mfcc(signal, sample_rate=SAMPLE_RATE, n_mfcc=N_MFCC, n_fft=N_FFT, hop_length=HOP_LENGTH):
    signal = np.asarray(signal, dtype=np.float32)
    padding = n_fft // 2
    padded_signal = np.pad(signal, (padding, padding), mode='reflect')

    f, t, Zxx = stft(
        padded_signal,
        fs=sample_rate,
        window='hann',
        nperseg=n_fft,
        noverlap=n_fft - hop_length,
        boundary=None,
        padded=False
    )
    
    amplitudes = np.abs(Zxx)
    spectrogram = amplitudes * HANN_SUM
    spectrogram = spectrogram.T
    
    mel_spectrogram = np.dot(spectrogram, MEL_FILTER_WEIGHTS)
    log_mel_spectrogram = np.log(mel_spectrogram + 1e-6)

    mfcc = dct(log_mel_spectrogram, type=2, axis=-1, norm='ortho')
    mfcc = mfcc[..., :n_mfcc]
  
    mfcc = mfcc.T
    mfcc = np.expand_dims(mfcc, axis=0)
    mfcc = np.expand_dims(mfcc, axis=-1)

    return mfcc.astype(np.float32)

def load_and_preprocess_wav(filepath):
    sr, audio = wavfile.read(filepath)
    
    if audio.dtype == np.int16:
        audio = audio.astype(np.float32) / 32768.0
    elif audio.dtype == np.int32:
        audio = audio.astype(np.float32) / 2147483648.0
    elif audio.dtype == np.uint8:
        audio = (audio.astype(np.float32) - 128.0) / 128.0
    else:
        audio = audio.astype(np.float32)

    if audio.ndim > 1:
        audio = audio[:, 0]

    if sr != SAMPLE_RATE:
        audio = resample_poly(audio, SAMPLE_RATE, sr).astype(np.float32)

    if len(audio) > SAMPLE_RATE:
        audio = audio[:SAMPLE_RATE]
    elif len(audio) < SAMPLE_RATE:
        audio = np.pad(audio, (0, SAMPLE_RATE - len(audio)), mode='constant')

    audio -= np.mean(audio)
    
    return audio

def predict_audio_signal(audio_signal):
    features = extract_mfcc(audio_signal)
    mfcc_int32 = np.clip(np.round(features * FIXED_SCALE_IN), -2147483648, 2147483647).astype(np.int32)

    input_buffer[:] = mfcc_int32.flatten(order='C')
    output_buffer[:] = 0

    my_ip.write(0x00, 1)
    
    dma.recvchannel.transfer(output_buffer)
    dma.sendchannel.transfer(input_buffer)
    dma.sendchannel.wait()
    dma.recvchannel.wait()

    output_data = np.array(output_buffer, dtype=np.int32)
    model_output = np.zeros(7, dtype=np.float32)

    for i in range(7):
        bit_offset = i * 24
        word_idx = bit_offset // 32
        shift = bit_offset % 32

        val = (output_data[word_idx] >> shift) & 0xFFFFFF

        if shift > 8:
            bits1 = 32 - shift
            bits2 = 24 - bits1
            mask1 = (1 << bits1) - 1
            mask2 = (1 << bits2) - 1

            part1 = (output_data[word_idx] >> shift) & mask1
            part2 = output_data[word_idx + 1] & mask2
            val = part1 | (part2 << bits1)

        if val & 0x800000:
            val -= 0x1000000

        model_output[i] = val

    logits = model_output.astype(np.float32) / FIXED_SCALE_OUT
    logits[1] -= UNKNOWN_LOGIT_OFFSET
    logits_shifted = logits - np.max(logits)

    probabilities = np.exp(logits_shifted) / np.sum(np.exp(logits_shifted))
    pred_idx = int(np.argmax(probabilities))
    
    return LABELS[pred_idx], probabilities


def evaluate(test_folder=TEST_FOLDER):
    label_true = []
    label_prediction = []

    all_files = []
    for label in LABELS:
        label_dir = os.path.join(test_folder, label)
        if os.path.exists(label_dir):
            files = [os.path.join(label_dir, f) for f in os.listdir(label_dir) if f.lower().endswith('.wav')]
            for f in files:
                all_files.append((label, f))

    total_files = len(all_files)

    for idx, (true_label, filepath) in enumerate(all_files, 1):
        try:
            audio_signal = load_and_preprocess_wav(filepath)
            predicted_label, _ = predict_audio_signal(audio_signal)

            label_true.append(true_label)
            label_prediction.append(predicted_label)

        except Exception as e:
            print(f"\Error pri obradi datoteke {filepath}: {e}")
            continue

        if idx % 50 == 0 or idx == total_files:
            elapsed = time.time() - start_eval_time
            fps = idx / elapsed if elapsed > 0 else 0
            print(f"\rObrađeno: [{idx}/{total_files}] ({idx/total_files*100:.1f}%) | Brzina: {fps:.1f} uzoraka/s", end="", flush=True)

    matrica = confusion_matrix(label_true, label_prediction, labels=LABELS)

    rows = []
    columns = []
    for label in LABELS:
        rows.append(f"Točno: {label}")
        columns.append(f"Predviđeno: {label}")

    matrica_view = pd.DataFrame(matrica, index=rows, columns=columns)
    print("\n")
    print(matrica_view)
    
    report = classification_report(label_true, label_prediction, labels=LABELS, target_names=LABELS, digits=3, zero_division=0)
    print("\n")
    print(report)

    txt_filename = "rezultati_testni_skup.txt"
    
    with open(txt_filename, "w", encoding="utf-8") as f:
        f.write("Matrica zabune\n")
        f.write(matrica_view.to_string() + "\n\n")
        f.write("Metrike klasifikacije\n")
        f.write(report + "\n\n")

    img_filename = "matrica_zabune_testni_skup.png"
    fig, ax = plt.subplots(figsize=(8, 7))
    im = ax.imshow(matrica, interpolation='nearest', cmap=plt.cm.Blues)
    fig.colorbar(im, ax=ax)

    ax.set(
        xticks=np.arange(matrica.shape[1]),
        yticks=np.arange(matrica.shape[0]),
        xticklabels=LABELS, 
        yticklabels=LABELS,
        ylabel='Stvarna klasa',
        xlabel='Predviđena klasa'
    )

    plt.setp(ax.get_xticklabels(), rotation=45, ha="right", rotation_mode="anchor")

    thresh = matrica.max() / 2.
    for i in range(matrica.shape[0]):
        for j in range(matrica.shape[1]):
            ax.text(
                j, i, format(matrica[i, j], 'd'),
                ha="center", va="center",
                color="white" if matrica[i, j] > thresh else "black"
            )

    fig.tight_layout()
    
    plt.savefig(img_filename, dpi=300)
    plt.show()
    plt.close()

    return label_true, label_prediction, matrica_view

label_true_test, label_prediction_test, matrica_test = evaluate()